# Crypto Price & Volume Data Pipeline

Fetches historical OHLCV (Open, High, Low, Close, Volume) data for all active
Binance USDT spot pairs and saves consolidated close-price and volume tables
for downstream analysis.
## Steps
1. **Retrieve symbols** — Get all active USDT spot pair tickers from Binance,
   ranked by quote (USDT) volume.
2. **Fetch OHLCV** — Download price/volume data for each symbol across multiple
   timeframes (1d, 8h, 4h, 1h), paginating back to `START_DATE` (or the coin's
   listing date, whichever is later).
3. **Cache** — Store each symbol/timeframe locally as Parquet so re-runs don't
   re-download.
4. **Assemble** — Build per-timeframe close-price and volume DataFrames
   (one column per coin).
5. **Save** — Write the consolidated price and volume tables to `data_output/`
   as pickle files.
## Output
- `data_cache/<SYMBOL>_<tf>.parquet` — raw per-symbol cache
- `data_output/crypto_px_<tf>.pkl` — close prices (rows = timestamps, cols = coins)
- `data_output/crypto_vol_<tf>.pkl` — volumes (same shape)


In [1]:
# _____ Imports _____
import ccxt
import pandas as pd
import time
from pathlib import Path
# ____ CONFIGURATION ____

TIMEFRAMES = ["1d", "8h", "4h", "1h"] # Timeframes to fetch
START_DATE = "2022-01-01" # Earliest date to request

DATA_DIR = "data_cache" # Parquet - efficient data storage
OUTPUT_DIR = "data_output" # Where the final pickle will live

FORCE_REFRESH = False # Set True to refetch everything
LIMIT_SYMBOLS = None   # e.g. 50 for testing, None = all Binance

# _____ Create folder ____
Path(DATA_DIR).mkdir(exist_ok = True)
Path(OUTPUT_DIR).mkdir(exist_ok=True)

In [2]:
# ____Binance spot exchange handle___
def get_bin_exchange():
    return ccxt.binance({
        "enableRateLimit": True,"options": {"defaultType": "spot"}})

In [3]:
# ___ Grabbing all/limited relevant active USDT coin symbols ___
def get_all_usdt_symbols():
    exchange = get_bin_exchange()
    markets = exchange.load_markets() # Load market metadata
    tickers = exchange.fetch_tickers() # Fetch all tickers
    
    filtered = {symbol:ticker # Filter only active USDT spot pairs
                for symbol, ticker in tickers.items()
                if (symbol.endswith('/USDT') and symbol in markets 
                    and markets[symbol]['active'] and markets[symbol]['spot'])}

    # Sort by quote volume (descending)
    symbols_sorted = sorted(
        filtered.keys(),key=lambda s: filtered[s]['quoteVolume'] or 0,
        reverse=True)

    # Apply optional limit (for testing)
    if LIMIT_SYMBOLS:
        symbols_sorted = symbols_sorted[:LIMIT_SYMBOLS]
    
    print(f"Using {len(symbols_sorted)} symbols")
    return symbols_sorted

In [4]:
# ____ Fetching price/volume data from Binance, with pagination ____
def fetch_ohlcv(symbol, timeframe, since):
    exchange = get_bin_exchange()
    markets = exchange.load_markets()
    # Never request data before the coin was actually listed
    symbol_info = markets.get(symbol, {}).get('info', {}) 
    onboard_date = symbol_info.get('onboardDate')
    if onboard_date is not None and str(onboard_date).isdigit():
        # Binance gives onboardDate in milliseconds
        symbol_launch = pd.to_datetime(int(onboard_date), unit='ms') #Binance gives ms
    else:
        # fallback if missing or not numeric
        symbol_launch = pd.Timestamp(since)
    actual_start = max(pd.Timestamp(since), symbol_launch) # Adjust start date so we never request data before listing
    since_ms = int(actual_start.timestamp() * 1000) # Convert to milliseconds for binance
    all_data = []
    max_pages = 1000 # Safety cap: stop after ~1m rows
    while True:
        data = exchange.fetch_ohlcv(symbol, timeframe, since=since_ms, limit=1000) # Grab data
        if not data:
            break
        all_data.extend(data)
        since_ms = data[-1][0] + 1 # Next page starts just after the last timestamp
        if len(data) < 1000: # Short page => we've reached the end
            break
        if len(all_data) >= max_pages * 1000:
            break
        
        time.sleep(0.2) 
    df = pd.DataFrame(all_data, columns=["ts","open","high","low","close","volume"])
    df["ts"] = pd.to_datetime(df["ts"], unit="ms")
    df = df.set_index("ts").sort_index()
    return df

In [5]:
# ____ Where cached data will live ____
def get_cache_path(symbol, timeframe):
    fname = symbol.replace("/", "_")
    return Path(DATA_DIR) / f"{fname}_{timeframe}.parquet"

# ____ Loading from cache or fetch ____
def load_or_fetch(symbol, timeframe):
    path = get_cache_path(symbol, timeframe)
    
    if path.exists() and not FORCE_REFRESH: #I.e. Check if cached already exists
        return pd.read_parquet(path)
    
    df = fetch_ohlcv(symbol, timeframe, START_DATE) # Otherwise, redownload data
    df.to_parquet(path)
    
    return df

In [6]:
# ____ Building price and volume dataframes ____
def build_dataset(symbols, timeframe):
    price_data = {}
    vol_data = {}
    
    for sym in symbols:
        try:
            df = load_or_fetch(sym, timeframe)
            price_data[sym] = df["close"]
            vol_data[sym] = df["volume"]
        except Exception as e:
            print(f"Error with {sym}: {e}")
    
    px = pd.DataFrame(price_data).sort_index()
    volx = pd.DataFrame(vol_data).sort_index()
    return px, volx

In [8]:
# ___ Save as pickle files ____
def save_output(px, volx, timeframe):

    px_path = Path(OUTPUT_DIR) / f"crypto_px_{timeframe}.pkl"
    vol_path = Path(OUTPUT_DIR) / f"crypto_vol_{timeframe}.pkl"

    px.to_pickle(px_path)
    volx.to_pickle(vol_path)

    print(f"Saved prices: {px_path}")
    print(f"Saved volumes: {vol_path}")

In [9]:
# ____ Updated pipeline code for all timeframes____
def run_pipeline():

    symbols = get_all_usdt_symbols()

    for tf in TIMEFRAMES:

        print(f"\nProcessing timeframe: {tf}")

        # Build both datasets
        px, volx = build_dataset(symbols, tf)

        # Save both
        save_output(px, volx, tf)

In [10]:
run_pipeline()

Using 435 symbols

Processing timeframe: 1d
Saved prices: data_output/crypto_px_1d.pkl
Saved volumes: data_output/crypto_vol_1d.pkl

Processing timeframe: 8h
Saved prices: data_output/crypto_px_8h.pkl
Saved volumes: data_output/crypto_vol_8h.pkl

Processing timeframe: 4h
Saved prices: data_output/crypto_px_4h.pkl
Saved volumes: data_output/crypto_vol_4h.pkl

Processing timeframe: 1h
Saved prices: data_output/crypto_px_1h.pkl
Saved volumes: data_output/crypto_vol_1h.pkl
